# Stratified Random Sample — FEC Legislators

Takes the matched FEC/legislator CSV and draws a stratified random sample
balanced across **party** and **small-donor quartile**.

Senate is small enough (100 members) that we keep all senators and only
sample from the House.

**Steps**
1. Load matched data
2. Filter & inspect
3. Configure sample size
4. Draw stratified sample
5. Inspect & export

---
## 0 · Imports & config

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 50)
pd.set_option("display.max_rows", 120)

In [ ]:
# ── Paths ───────────────────────────────────────────────────────────────────
MATCHED_CSV  = Path("fec_legislators_matched.csv")
OUTPUT_CSV   = Path("sample.csv")

# ── Sampling config ─────────────────────────────────────────────────────────
RANDOM_SEED  = 42    # change to get a different draw; keep fixed for reproducibility

# How many House members to sample per stratum (party x quartile).
# With 2 parties x 4 quartiles = 8 strata, N_PER_STRATUM=8 gives ~64 House members.
# The actual count may be lower if a stratum has fewer members than requested.
N_PER_STRATUM = 8

# Set to True to keep ALL senators (recommended — only 100 of them).
KEEP_ALL_SENATORS = True

---
## 1 · Load matched data

In [ ]:
df = pd.read_csv(MATCHED_CSV)
print(f"Total rows: {len(df)}")
print()
print("Match method breakdown:")
print(df["match_method"].value_counts())
df.head()

---
## 2 · Filter & inspect

We drop:
- Rows with no bioguide (true non-members / candidates who never served)
- Rows with no `pct_small_donors` (can't stratify without the key variable)
- Members who raised less than $100k (noisy small-donor percentages)

In [ ]:
# Keep only matched members
pool = df[df["bioguide"].notna()].copy()
print(f"After dropping non-members: {len(pool)}")

# Drop rows missing the pct column
pool = pool[pool["pct_small_donors"].notna()]
print(f"After dropping missing pct_small_donors: {len(pool)}")

# Drop low fundraisers (noisy percentages)
if "total money raised" in pool.columns:
    pool = pool[pool["total money raised"] == "Raised over $100k"]
    print(f"After dropping <$100k raisers: {len(pool)}")

# Normalize party to R/D/I for stratification
pool["party_code"] = pool["party_code"].fillna(
    pool["party"].map({"Republican": "R", "Democrat": "D", "Independent": "I"})
)

print()
print("Pool breakdown:")
print(pool.groupby(["chamber", "party_code"]).size().unstack(fill_value=0))

In [ ]:
# Distribution of small-donor % across the pool
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

pool["pct_small_donors"].hist(bins=20, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("All members: small-donor %")
axes[0].set_xlabel("% from small donors")
axes[0].set_ylabel("Count")

for party, color in [("R", "tomato"), ("D", "steelblue")]:
    subset = pool[pool["party_code"] == party]["pct_small_donors"]
    subset.hist(bins=20, ax=axes[1], alpha=0.6, label=party, color=color, edgecolor="white")
axes[1].set_title("By party: small-donor %")
axes[1].set_xlabel("% from small donors")
axes[1].legend()

plt.tight_layout()
plt.show()

print(pool.groupby("party_code")["pct_small_donors"].describe().round(1))

---
## 3 · Assign quartiles

Quartiles are computed **within each chamber separately** so House and Senate
members are compared to their own chamber peers, not to each other
(senators generally raise very differently from House members).

In [ ]:
def assign_quartiles(group):
    group = group.copy()
    group["quartile"] = pd.qcut(
        group["pct_small_donors"],
        q=4,
        labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"],
        duplicates="drop",
    )
    return group

pool = pool.groupby("chamber", group_keys=False).apply(assign_quartiles)

print("Quartile boundaries by chamber:")
for chamber, grp in pool.groupby("chamber"):
    print(f"  {chamber}:")
    print(grp.groupby("quartile")["pct_small_donors"].agg(["min", "max", "count"]).round(1))
    print()

---
## 4 · Draw stratified sample

In [ ]:
house  = pool[pool["chamber"] == "rep"].copy()
senate = pool[pool["chamber"] == "sen"].copy()

# ── Senate: keep all (or sample if you prefer) ───────────────────────────
if KEEP_ALL_SENATORS:
    senate_sample = senate.copy()
    senate_sample["sample_method"] = "all senators kept"
    print(f"Senators kept: {len(senate_sample)}")
else:
    senate_sample = (
        senate
        .groupby(["party_code", "quartile"], group_keys=False)
        .apply(lambda x: x.sample(min(len(x), N_PER_STRATUM), random_state=RANDOM_SEED))
    )
    senate_sample["sample_method"] = "stratified"
    print(f"Senate sample: {len(senate_sample)}")

# ── House: stratified by party x quartile ────────────────────────────────
house_sample = (
    house
    .groupby(["party_code", "quartile"], group_keys=False)
    .apply(lambda x: x.sample(min(len(x), N_PER_STRATUM), random_state=RANDOM_SEED))
)
house_sample["sample_method"] = "stratified"
print(f"House sample : {len(house_sample)}")

# ── Combine ───────────────────────────────────────────────────────────────
sample = pd.concat([senate_sample, house_sample], ignore_index=True)
print(f"\nTotal sample : {len(sample)}")

---
## 5 · Inspect sample

In [ ]:
print("=== Sample composition ===")
print()
print("By chamber x party:")
print(sample.groupby(["chamber", "party_code"]).size().unstack(fill_value=0))
print()
print("House strata counts (party x quartile):")
house_s = sample[sample["chamber"] == "rep"]
print(house_s.groupby(["party_code", "quartile"]).size().unstack(fill_value=0))

In [ ]:
# Compare pct_small_donors distribution: pool vs sample
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for party, color in [("R", "tomato"), ("D", "steelblue")]:
    pool[pool["party_code"] == party]["pct_small_donors"].hist(
        bins=20, ax=axes[0], alpha=0.6, label=party, color=color, edgecolor="white")
    sample[sample["party_code"] == party]["pct_small_donors"].hist(
        bins=20, ax=axes[1], alpha=0.6, label=party, color=color, edgecolor="white")

axes[0].set_title("Full pool")
axes[1].set_title("Sample")
for ax in axes:
    ax.set_xlabel("% from small donors")
    ax.legend()
plt.tight_layout()
plt.show()

print("Pool stats:")
print(pool.groupby("party_code")["pct_small_donors"].describe().round(1))
print()
print("Sample stats:")
print(sample.groupby("party_code")["pct_small_donors"].describe().round(1))

In [ ]:
# Twitter handle coverage — important for your tweet collection step
has_twitter = sample["twitter"].notna() & (sample["twitter"] != "")
print(f"Twitter handles available: {has_twitter.sum()} / {len(sample)} ({100*has_twitter.mean():.0f}%)")
print()
print("Members WITHOUT a twitter handle:")
no_twitter = sample[~has_twitter][["official_full", "chamber", "party_code", "state"]]
display(no_twitter)

In [ ]:
# Full sample list
sample[["official_full", "twitter", "chamber", "state", "party_code",
        "quartile", "pct_small_donors"]].sort_values(
    ["chamber", "party_code", "quartile"]
)

---
## 6 · Export

In [ ]:
sample.to_csv(OUTPUT_CSV, index=False)
print(f"Saved → {OUTPUT_CSV}  ({len(sample)} rows)")
print(f"Seed used: {RANDOM_SEED}  — change RANDOM_SEED in cell 0 for a different draw")